# <font color="#418FDE" size="6.5" uppercase>**Merkmale erklären**</font>

>Last update: 20260723.
    
By the end of this Lecture, you will be able to:
- Konstruieren und wählen Merkmale innerhalb von scikit-learn-Pipelines aus. 
- Vergleichen Dimensionsreduktion und Merkmalsauswahl für unterschiedliche Datenformen. 
- Interpretieren Modelle mit Koeffizienten, Importances und Abhängigkeitsplots. 


## **1. Merkmale konstruieren**

### **1.1. Polynomiale Merkmale**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_11/Lecture_B/image_01_01.jpg?v=1784801297" width="250">



>* Neue Merkmale aus Potenzen und Kombinationen
>* Lineare Modelle lernen gekrümmte Zusammenhänge

>* Polynomiale Merkmale in Pipelines skalieren
>* Trainingstransformation verhindert Datenleckage beim Tuning

>* Nützlich bei passenden, gut verstandenen Daten
>* Mit Validierung gegen Überanpassung absichern



In [ ]:
#@title Python-Code - Polynomiale Merkmale

# Dieses Beispiel zeigt polynomiale Merkmale in Pipelines.
# Ein lineares Modell lernt dadurch gekrümmte Muster.
# Die Ausgabe vergleicht einfache und erweiterte Merkmale.

import numpy as np
import matplotlib.pyplot as plt
from sklearn import __version__ as sklearn_version
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures

# Wir erzeugen kleine Daten mit einem gekrümmten Zusammenhang.
rng = np.random.default_rng(42)
x = np.linspace(-3, 3, 80).reshape(-1, 1)
noise = rng.normal(0, 1.2, size=x.shape[0])
y = 2.0 + 0.8 * x.ravel() + 1.5 * x.ravel() ** 2 + noise

# Die Aufteilung verhindert Bewertung auf bereits gesehenen Daten.
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.25, random_state=42
)

# Eine einfache Prüfung macht die erwartete Datenform sichtbar.
if X_train.shape[1] != 1:
    raise ValueError("Erwartet wird genau ein numerisches Merkmal.")

# Dieses Modell nutzt nur das ursprüngliche Merkmal.
linear_model = Pipeline([
    ("model", LinearRegression())
])
linear_model.fit(X_train, y_train)

# Dieses Modell konstruiert zusätzlich x² vor der Regression.
poly_model = Pipeline([
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("model", LinearRegression())
])
poly_model.fit(X_train, y_train)

# Wir bewerten beide Pipelines auf denselben Testdaten.
linear_rmse = np.sqrt(mean_squared_error(
    y_test, linear_model.predict(X_test)
))
poly_rmse = np.sqrt(mean_squared_error(
    y_test, poly_model.predict(X_test)
))

# Die Namen zeigen, welche Merkmale die Pipeline erzeugt.
feature_names = poly_model.named_steps["poly"].get_feature_names_out(["x"])
print(f"scikit-learn-Version: {sklearn_version}")
print(f"Erzeugte Merkmale: {', '.join(feature_names)}")
print(f"RMSE ohne polynomiale Merkmale: {linear_rmse:.2f}")
print(f"RMSE mit polynomialen Merkmalen: {poly_rmse:.2f}")

# Die Kurven zeigen den Unterschied der gelernten Zusammenhänge.
x_grid = np.linspace(-3.2, 3.2, 200).reshape(-1, 1)
linear_curve = linear_model.predict(x_grid)
poly_curve = poly_model.predict(x_grid)

# Eine einzige Grafik hält den Vergleich übersichtlich.
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(X_test.ravel(), y_test, label="Testdaten", alpha=0.75)
ax.plot(x_grid.ravel(), linear_curve, label="Linear", linewidth=2)
ax.plot(x_grid.ravel(), poly_curve, label="Polynomial Grad 2", linewidth=2)

# Beschriftungen machen die Bedeutung der Achsen klar.
ax.set_title("Polynomiale Merkmale in einer scikit-learn-Pipeline")
ax.set_xlabel("Eingabemerkmal x")
ax.set_ylabel("Zielwert y")
ax.legend()
plt.show()



### **1.2. Binning und Splines**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_11/Lecture_B/image_01_02.jpg?v=1784801299" width="250">



>* Binning ordnet Zahlenwerten sinnvolle Bereiche zu
>* Pipeline-Nutzung verhindert Leckage, kostet aber Detailinformation

>* Splines modellieren glatte nichtlineare Zusammenhänge
>* Lineare Modelle bleiben interpretierbar und flexibler

>* Pipelines machen Transformationen vergleichbar und reproduzierbar
>* Mehr Merkmale erfordern sorgfältige Validierung



In [ ]:
#@title Python-Code - Binning und Splines

# Dieses Beispiel vergleicht Binning und Splines.
# Beide Transformationen entstehen innerhalb einer Pipeline.
# Die Grafik zeigt gelernte nichtlineare Vorhersagen.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.preprocessing import SplineTransformer

# Wir erzeugen kleine Trainingsdaten mit gekrümmtem Zusammenhang.
rng = np.random.default_rng(42)
x_train = np.linspace(0, 10, 80).reshape(-1, 1)
noise = rng.normal(0, 0.25, size=x_train.shape[0])

# Die Zielwerte steigen und fallen nicht streng linear.
y_train = np.sin(x_train[:, 0]) + 0.15 * x_train[:, 0] + noise
x_grid = np.linspace(0, 10, 300).reshape(-1, 1)

# Eine einfache Prüfung macht die Datenannahme sichtbar.
if x_train.shape[0] != y_train.shape[0]:
    raise ValueError("Merkmale und Zielwerte müssen gleich viele Zeilen haben.")

# Binning erzeugt harte Bereiche vor dem linearen Modell.
binned_model = make_pipeline(
    KBinsDiscretizer(n_bins=8, encode="onehot-dense", strategy="quantile"),
    LinearRegression(),
)

# Splines erzeugen glatte Basisfunktionen vor dem linearen Modell.
spline_model = make_pipeline(
    SplineTransformer(n_knots=6, degree=3, include_bias=False),
    LinearRegression(),
)

# Beide Pipelines lernen Transformation und Modell gemeinsam.
binned_model.fit(x_train, y_train)
spline_model.fit(x_train, y_train)

# Die Vorhersagen zeigen den Unterschied der Merkmalskonstruktion.
y_binned = binned_model.predict(x_grid)
y_spline = spline_model.predict(x_grid)

print(f"scikit-learn-Version: {sklearn.__version__}")
print("Binning: stufige Vorhersage durch diskrete Bereiche.")
print("Splines: glattere Vorhersage durch überlappende Basisfunktionen.")

# Eine einzige Grafik vergleicht Daten und beide Pipeline-Varianten.
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x_train[:, 0], y_train, s=18, alpha=0.55, label="Trainingsdaten")
ax.plot(x_grid[:, 0], y_binned, label="Pipeline mit Binning")
ax.plot(x_grid[:, 0], y_spline, label="Pipeline mit Splines")

ax.set_title("Binning und Splines als Merkmalskonstruktion")
ax.set_xlabel("Eingangsmerkmal x")
ax.set_ylabel("Vorhergesagter Zielwert")
ax.legend()
plt.show()



### **1.3. Varianz und KBest**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_11/Lecture_B/image_01_03.jpg?v=1784801301" width="250">



>* Varianzfilter entfernen kaum veränderliche Merkmale
>* Pipelines machen die Auswahl reproduzierbar

>* Varianz allein zeigt keine Zielrelevanz
>* KBest bewertet Merkmale einzeln statistisch

>* Auswahl hält erzeugte Merkmale beherrschbar
>* Pipeline verhindert Datenleckage bei Kreuzvalidierung



In [ ]:
#@title Python-Code - Varianz und KBest

# Wir wählen Merkmale direkt in einer Pipeline.
# Varianzfilter entfernt fast konstante Spalten zuerst.
# KBest behält danach zielnahe Merkmale.

import numpy as np
import sklearn
from sklearn.datasets import make_classification
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import VarianceThreshold
from sklearn.feature_selection import f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import StandardScaler

# Ein kleiner Datensatz enthält nützliche und nutzlose Merkmale.
X, y = make_classification(
    n_samples=300,
    n_features=5,
    n_informative=2,
    n_redundant=0,
    random_state=42,
)

# Eine fast konstante Spalte simuliert ein schwaches Merkmal.
almost_constant = np.zeros((X.shape[0], 1))
almost_constant[:3, 0] = 1
X = np.hstack([X, almost_constant])

# Diese Prüfung macht die erwartete Tabellenform sichtbar.
if X.shape != (300, 6):
    raise ValueError("Die Beispieldaten haben eine unerwartete Form.")

# Die Aufteilung verhindert Datenleckage beim späteren Auswählen.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    stratify=y,
    random_state=42,
)

# Die Pipeline konstruiert, filtert, wählt und trainiert gemeinsam.
pipeline = Pipeline(
    steps=[
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("variance", VarianceThreshold(threshold=0.01)),
        ("kbest", SelectKBest(score_func=f_classif, k=8)),
        ("scale", StandardScaler()),
        ("model", LogisticRegression(max_iter=500, random_state=42)),
    ]
)

# Alle Auswahlregeln werden nur auf Trainingsdaten gelernt.
pipeline.fit(X_train, y_train)
accuracy = pipeline.score(X_test, y_test)

# Wir zählen, wie viele Spalten jeder Schritt übrig lässt.
poly_features = pipeline.named_steps["poly"].get_feature_names_out()
after_variance = pipeline.named_steps["variance"].get_support()
after_kbest = pipeline.named_steps["kbest"].get_support()
selected_names = poly_features[after_variance][after_kbest]

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Nach PolynomialFeatures: {len(poly_features)} Merkmale")
print(f"Nach VarianceThreshold: {int(after_variance.sum())} Merkmale")
print(f"Nach SelectKBest: {len(selected_names)} Merkmale")
print(f"Testgenauigkeit der Pipeline: {accuracy:.3f}")
print("Ausgewählte Merkmale: " + ", ".join(selected_names[:8]))



## **2. Auswahl und Reduktion**

### **2.1. Rekursive Merkmalsauswahl**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_11/Lecture_B/image_02_01.jpg?v=1784801289" width="250">



>* Schrittweise unwichtige Merkmale entfernen
>* Auswahl hängt vom verwendeten Modell ab

>* Erhält Bedeutung, verbessert Interpretierbarkeit
>* Kostet Rechenzeit, reagiert auf Korrelationen

>* Gut für schlanke, verständliche Tabellenmodelle
>* Datenform, Aufwand und Stabilität mitbedenken



In [ ]:
#@title Python-Code - Rekursive Merkmalsauswahl

# Dieses Beispiel zeigt rekursive Merkmalsauswahl.
# RFE entfernt schrittweise weniger hilfreiche Merkmale.
# Die Ausgabe vergleicht Auswahl und Modellgüte.

import numpy as np
import pandas as pd
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Wir nutzen einen kleinen tabellarischen Klassifikationsdatensatz.
data = load_breast_cancer()
X = data.data
feature_names = data.feature_names
y = data.target

# Eine kurze Prüfung macht die Annahmen sichtbar.
if X.shape[0] != y.shape[0]:
    raise ValueError("Merkmale und Zielwerte passen nicht zusammen.")

# Die Aufteilung verhindert Datenleckage beim Skalieren und Auswählen.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

# RFE wählt fünf ursprüngliche Merkmale für Logistic Regression aus.
rfe_model = LogisticRegression(max_iter=1000, random_state=42)
selector = RFE(estimator=rfe_model, n_features_to_select=5, step=1)

# Die Pipeline passt Skalierung, Auswahl und Modell nur auf Trainingsdaten an.
pipeline = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("rfe", selector),
        ("model", LogisticRegression(max_iter=1000, random_state=42)),
    ]
)
pipeline.fit(X_train, y_train)

# Die ausgewählten Spalten behalten ihre ursprünglichen Namen.
selected_mask = pipeline.named_steps["rfe"].support_
selected_features = feature_names[selected_mask]

# Wir vergleichen ein schlankes Modell mit einem Modell auf allen Merkmalen.
full_pipeline = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, random_state=42)),
    ]
)
full_pipeline.fit(X_train, y_train)

# Beide Genauigkeiten werden auf denselben Testdaten gemessen.
selected_accuracy = accuracy_score(y_test, pipeline.predict(X_test))
full_accuracy = accuracy_score(y_test, full_pipeline.predict(X_test))

# Eine kleine Tabelle zeigt die interpretierbare Merkmalsauswahl.
result_table = pd.DataFrame({"ausgewählte Merkmale": selected_features})
print("scikit-learn-Version:", sklearn.__version__)
print("Genauigkeit mit allen 30 Merkmalen:", round(full_accuracy, 3))
print("Genauigkeit mit 5 RFE-Merkmalen:", round(selected_accuracy, 3))
print(result_table.to_string(index=False))



### **2.2. PCA und SVD**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_11/Lecture_B/image_02_02.jpg?v=1784801287" width="250">



>* PCA und SVD verdichten viele Merkmale
>* Neue Komponenten reduzieren Rauschen, erschweren Interpretation

>* PCA für skalierte, dichte Zahlentabellen
>* SVD für große, dünne Textmatrizen

>* Merkmalsauswahl erleichtert fachliche Interpretierbarkeit
>* PCA/SVD für Leistung, Kompression und Pipelines



In [ ]:
#@title Python-Code - PCA und SVD

# Dieses Beispiel vergleicht PCA und SVD.
# Beide Verfahren reduzieren viele Merkmale kompakt.
# Die Ausgabe zeigt passende Datenformen.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_wine
from sklearn.decomposition import PCA
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import StandardScaler

# Dichte Messdaten eignen sich gut für PCA.
wine = load_wine()
X_dense = wine.data

# Diese Prüfung macht die Datenannahme sichtbar.
if X_dense.shape[1] < 3:
    raise ValueError("Für PCA werden mehrere numerische Merkmale benötigt.")

# Skalierung verhindert Dominanz großer Zahlenbereiche.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_dense)

# PCA erzeugt zwei neue Komponenten aus allen Spalten.
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# Kurze Texte erzeugen eine dünn besetzte Wortmatrix.
documents = [
    "maschine temperatur vibration druck",
    "maschine vibration strom druck",
    "kunde kauf produkt rabatt",
    "kunde produkt bewertung kauf",
    "labor blutwert risiko medizin",
    "medizin risiko diagnose labor",
]

# CountVectorizer baut viele Wortspalten mit vielen Nullen.
vectorizer = CountVectorizer()
X_sparse = vectorizer.fit_transform(documents)

# SVD reduziert die dünn besetzte Matrix ohne Zentrierung.
svd = TruncatedSVD(n_components=2, random_state=42)
X_svd = svd.fit_transform(X_sparse)

# Die Kennzahlen vergleichen erklärte Struktur und Datenform.
print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"PCA auf dichter Tabelle: {X_dense.shape[1]} -> 2 Merkmale")
print(f"Erklärte Varianz durch PCA: {pca.explained_variance_ratio_.sum():.2f}")
print(f"SVD auf dünner Textmatrix: {X_sparse.shape[1]} -> 2 Merkmale")
print(f"Erklärte Varianz durch SVD: {svd.explained_variance_ratio_.sum():.2f}")

# Ein Streudiagramm zeigt die neue zweidimensionale PCA-Sicht.
fig, ax = plt.subplots(figsize=(6, 4))
scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=wine.target, cmap="viridis")

# Achsen und Titel beschreiben die reduzierten Komponenten.
ax.set_title("PCA: Wein-Daten in zwei Komponenten")
ax.set_xlabel("Hauptkomponente 1")
ax.set_ylabel("Hauptkomponente 2")

# Die Legende verbindet Farben mit den Klassen.
legend = ax.legend(*scatter.legend_elements(), title="Klasse")
ax.add_artist(legend)
plt.tight_layout()
plt.show()



### **2.3. Auswahl in Pipelines**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_11/Lecture_B/image_02_03.jpg?v=1784801285" width="250">



>* Pipelines ordnen Auswahl, Transformation und Training.
>* Sie verhindern Datenleckage bei der Bewertung.

>* Auswahl bleibt interpretierbar, Reduktion verdichtet Muster
>* Pipelines vergleichen Leistung, Aufwand und Stabilität

>* Pipelines stimmen Schritte und Parameter gemeinsam ab
>* Merkmalsanzahl beeinflusst Nutzbarkeit und Reproduzierbarkeit



In [ ]:
#@title Python-Code - Auswahl in Pipelines

# Dieses Beispiel vergleicht Auswahl und Reduktion.
# Beide Varianten stehen sauber in Pipelines.
# Die Ausgabe zeigt Genauigkeit und Interpretierbarkeit.

import numpy as np
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Wir laden einen kleinen tabellarischen Klassifikationsdatensatz.
data = load_breast_cancer()
X = data.data
y = data.target

# Eine einfache Prüfung macht die Annahmen sichtbar.
if X.shape[0] != y.shape[0] or X.shape[1] < 6:
    raise ValueError("Die Daten passen nicht zum Beispiel.")

# Die Aufteilung verhindert Datenleckage vor der Pipeline.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

# Diese Pipeline behält ausgewählte Originalmerkmale.
selection_pipeline = Pipeline(
    [
        ("scale", StandardScaler()),
        ("select", SelectKBest(score_func=f_classif, k=6)),
        ("model", LogisticRegression(max_iter=1000, random_state=42)),
    ]
)

# Diese Pipeline erzeugt sechs neue verdichtete Komponenten.
pca_pipeline = Pipeline(
    [
        ("scale", StandardScaler()),
        ("reduce", PCA(n_components=6, random_state=42)),
        ("model", LogisticRegression(max_iter=1000, random_state=42)),
    ]
)

# Beide Pipelines lernen Auswahl oder Reduktion nur aus Trainingsdaten.
selection_pipeline.fit(X_train, y_train)
pca_pipeline.fit(X_train, y_train)

# Wir bewerten beide Varianten auf denselben Testdaten.
selection_accuracy = accuracy_score(y_test, selection_pipeline.predict(X_test))
pca_accuracy = accuracy_score(y_test, pca_pipeline.predict(X_test))

# Die ausgewählten Namen bleiben fachlich direkt lesbar.
selected_mask = selection_pipeline.named_steps["select"].get_support()
selected_names = np.array(data.feature_names)[selected_mask]

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Merkmalsauswahl, Testgenauigkeit: {selection_accuracy:.3f}")
print(f"PCA-Reduktion, Testgenauigkeit: {pca_accuracy:.3f}")
print("Ausgewählte Originalmerkmale: " + ", ".join(selected_names[:6]))
print("PCA-Komponenten sind neue Mischmerkmale, keine Originalspalten.")



## **3. Modelle verständlich inspizieren**

### **3.1. Koeffizienten verstehen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_11/Lecture_B/image_03_01.jpg?v=1784801291" width="250">



>* Koeffizienten zeigen Richtung und Stärke
>* Vorhersagen werden dadurch nachvollziehbarer

>* Skalierung und Kodierung prägen Koeffizienten.
>* Korrelationen erschweren eindeutige Interpretationen.

>* Koeffizienten zeigen Zusammenhänge, keine sicheren Ursachen
>* Regularisierung und Kontext kritisch mitprüfen



In [ ]:
#@title Python-Code - Koeffizienten verstehen

# Wir untersuchen Koeffizienten eines linearen Modells.
# Standardisierung macht Koeffizienten besser vergleichbar.
# Ein Balkendiagramm zeigt Richtung und Stärke.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Wir laden einen kleinen Regressionsdatensatz aus scikit-learn.
diabetes = load_diabetes(as_frame=True)
features = diabetes.data

target = diabetes.target
feature_names = list(features.columns)

# Diese Prüfung macht die Annahmen des Beispiels sichtbar.
if features.shape[0] != len(target):
    raise ValueError("Merkmale und Zielwerte passen nicht zusammen.")

# Wir trennen Trainingsdaten und Testdaten ohne Datenleckage.
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.25, random_state=42
)

# Die Pipeline standardisiert zuerst und trainiert danach linear.
model = make_pipeline(StandardScaler(), LinearRegression())
model.fit(X_train, y_train)

# Nach Standardisierung sind Koeffizienten direkt vergleichbarer.
linear_model = model.named_steps["linearregression"]
coefficients = linear_model.coef_

coef_table = pd.DataFrame(
    {"Merkmal": feature_names, "Koeffizient": coefficients}
)

coef_table["Betrag"] = np.abs(coef_table["Koeffizient"])
coef_table = coef_table.sort_values("Betrag", ascending=False).head(5)

# Wir drucken nur wenige zentrale Lerninformationen.
score = model.score(X_test, y_test)
print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Test-R² des linearen Modells: {score:.2f}")
print("Positive Balken erhöhen die Vorhersage, negative senken sie.")

# Das Diagramm zeigt Richtung und relative Stärke der Koeffizienten.
fig, ax = plt.subplots(figsize=(8, 4))
colors = np.where(coef_table["Koeffizient"] >= 0, "tab:blue", "tab:orange")

ax.barh(coef_table["Merkmal"], coef_table["Koeffizient"], color=colors)
ax.axvline(0, color="black", linewidth=1)
ax.set_title("Größte standardisierte Koeffizienten")

ax.set_xlabel("Koeffizient nach Standardisierung")
ax.set_ylabel("Merkmal")
ax.invert_yaxis()
plt.tight_layout()

plt.show()



### **3.2. Merkmalswichtigkeit verstehen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_11/Lecture_B/image_03_02.jpg?v=1784801293" width="250">



>* Wichtige Merkmale prägen Modellvorhersagen stark
>* Wichtigkeit bedeutet nicht automatisch Kausalität

>* Modellinterne Wichtigkeiten sind schnell, aber verzerrbar
>* Permutation misst Leistungsverlust durch gestörte Merkmale

>* Wichtigkeit gilt nicht für jeden Einzelfall
>* Mit Fachwissen und Fairness kritisch prüfen



In [ ]:
#@title Python-Code - Merkmalswichtigkeit verstehen

# Wir untersuchen Merkmalswichtigkeit an einem kleinen Klassifikationsmodell.
# Permutation zeigt, welche Informationen die Leistung stützen.
# Ein Balkendiagramm macht die wichtigsten Merkmale sichtbar.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

# Wir laden einen kleinen, eingebauten Datensatz.
data = load_breast_cancer()
X = data.data
y = data.target

# Diese Prüfung macht die erwartete Tabellenform explizit.
if X.shape[0] != y.shape[0]:
    raise ValueError("Merkmale und Zielwerte passen nicht zusammen.")

# Der Split trennt Training und faire spätere Prüfung.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

# Ein kleiner Entscheidungsbaum bleibt gut nachvollziehbar.
model = DecisionTreeClassifier(max_depth=4, random_state=42)
model.fit(X_train, y_train)

# Die Genauigkeit zeigt zuerst die normale Modellleistung.
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

# Permutation misst den Leistungsverlust pro Merkmal.
result = permutation_importance(
    model, X_test, y_test, n_repeats=10, random_state=42
)

# Wir zeigen nur die fünf stärksten Merkmale.
importances = result.importances_mean
top_indices = np.argsort(importances)[-5:]
top_names = data.feature_names[top_indices]
top_values = importances[top_indices]

print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Testgenauigkeit ohne Vertauschen: {accuracy:.3f}")
print("Höhere Balken bedeuten stärkeren Leistungsverlust beim Vertauschen.")

# Das Diagramm ordnet die wichtigsten Merkmale übersichtlich.
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(top_names, top_values, color="steelblue")
ax.set_title("Permutationswichtigkeit der fünf stärksten Merkmale")
ax.set_xlabel("Mittlerer Genauigkeitsverlust")
ax.set_ylabel("Merkmal")
plt.tight_layout()
plt.show()



### **3.3. Erklärungsprojekt**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_11/Lecture_B/image_03_03.jpg?v=1784801295" width="250">



>* Erklärungsziel vor dem Training klären
>* Zielgruppe bestimmt passende Erklärungsmethode

>* Koeffizienten, Importances, Plots ergänzen sich
>* Erklärungen bleiben vorsichtige Interpretationen

>* Befunde verständlich und fachlich geprüft erzählen
>* Grenzen, Unsicherheiten und sensible Merkmale offenlegen



In [ ]:
#@title Python-Code - Erklärungsprojekt

# Dieses Projekt macht Modellvorhersagen verständlicher.
# Koeffizienten zeigen Richtung und relative Stärke.
# Ein Abhängigkeitsplot ergänzt die globale Interpretation.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_diabetes
from sklearn.inspection import PartialDependenceDisplay
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Wir nutzen einen kleinen Regressionsdatensatz aus scikit-learn.
diabetes = load_diabetes(as_frame=True)
X = diabetes.data
y = diabetes.target

# Eine einfache Prüfung verhindert missverständliche Folgefehler.
if X.shape[0] != len(y):
    raise ValueError("Merkmale und Zielwerte passen nicht zusammen.")

# Die Aufteilung trennt Training und faire Prüfung.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# Skalierung macht Koeffizienten besser vergleichbar.
model = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
model.fit(X_train, y_train)

# Die Testgüte zeigt, ob Erklärungen überhaupt nützlich sind.
y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)

# Koeffizienten beschreiben Richtung und Stärke im linearen Modell.
ridge_model = model.named_steps["ridge"]
coefficients = ridge_model.coef_
feature_names = np.array(X.columns)

# Wir sortieren nach absoluter Stärke der Koeffizienten.
order = np.argsort(np.abs(coefficients))[::-1]
top_features = feature_names[order[:3]]
top_values = coefficients[order[:3]]

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Test-R²: {r2:.2f}")
print("Stärkste Koeffizienten:")
for name, value in zip(top_features, top_values):
    print(f"{name}: {value:.1f}")

# Der Plot zeigt den durchschnittlichen Effekt eines wichtigen Merkmals.
fig, ax = plt.subplots(figsize=(7, 4))
PartialDependenceDisplay.from_estimator(
    model, X_train, [top_features[0]], ax=ax
)

ax.set_title("Abhängigkeitsplot für das wichtigste Merkmal")
ax.set_xlabel(f"Merkmal: {top_features[0]}")
ax.set_ylabel("Vorhergesagter Krankheitswert")
plt.tight_layout()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Merkmale erklären**</font>


In this lecture, you learned to:
- Konstruieren und wählen Merkmale innerhalb von scikit-learn-Pipelines aus. 
- Vergleichen Dimensionsreduktion und Merkmalsauswahl für unterschiedliche Datenformen. 
- Interpretieren Modelle mit Koeffizienten, Importances und Abhängigkeitsplots. 

In the next Module (Module 12), we will go over 'Cluster und Text'